# Use RapidAPI to pull company stock trading data

In [0]:
import requests
import pandas as pd
import pyspark.sql.functions as F
from functools import reduce
from io import StringIO
import time
# import time to delay execution of API calls to avoid overloading the API

url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "APIKEY"
}

## Pull stock data for each company

In [0]:
from pyspark.sql.functions import col, lit

# create function to extract csv data for each individual company
def extract_stock_data(company):
    querystring_time_series = {
    "function":"TIME_SERIES_DAILY",
    "symbol":f"{company}",
    "datatype":"csv",
    "outputsize":"compact"
    }
    metadata_query = {
        "function":"OVERVIEW",
        "symbol":f"{company}"
    }
    metadata_info = requests.get(url, headers=headers, params=metadata_query)
    metadata = metadata_info.json()
    exchange = metadata['Exchange']
    sector = metadata['Sector']

    response = requests.get(url, headers=headers, params=querystring_time_series)
    csv_text = response.text
    pd_df = pd.read_csv(StringIO(csv_text))
    spark_df = spark.createDataFrame(pd_df).orderBy('timestamp', ascending=True)
    spark_df_new = spark_df.withColumn('symbol', lit(company)).withColumn('exchange', lit(exchange)).withColumn('sector', lit(sector))
    return spark_df_new

## Update tables incrementally if they exist, create tables if they don't exist

In [0]:
%sql
USE jarvis_training_catalog.dltproject

In [0]:
def update_table(company):
    table_name = f"{company}_stock_data_bronze"
    company_stock_df = extract_stock_data(company)
    # pulls sorted stock data, including symbol, exchange, and sector
    if not spark.catalog.tableExists(table_name):
        company_stock_df.write.mode('overwrite').saveAsTable(table_name)
    else:
        prev_table = spark.table(table_name)
        new_rows = company_stock_df.exceptAll(prev_table)
        new_rows.write.mode('append').saveAsTable(table_name)

In [0]:
# our companies are Apple, Google, Microsoft, and Tesla
update_table('AAPL')
apple_table_bronze = spark.table('aapl_stock_data_bronze')
display(apple_table_bronze.head(5))

timestamp,open,high,low,close,volume,symbol,exchange,sector
2025-09-22,248.3,256.64,248.12,256.08,105517416,AAPL,NASDAQ,TECHNOLOGY
2025-09-23,255.875,257.34,253.58,254.43,60275187,AAPL,NASDAQ,TECHNOLOGY
2025-09-24,255.22,255.74,251.04,252.31,42303710,AAPL,NASDAQ,TECHNOLOGY
2025-09-25,253.205,257.17,251.712,256.87,55202075,AAPL,NASDAQ,TECHNOLOGY
2025-09-26,254.095,257.6,253.78,255.46,46076258,AAPL,NASDAQ,TECHNOLOGY


In [0]:
# our companies are Apple, Google, Microsoft, and Tesla
time.sleep(30)
update_table('GOOGL')
apple_table_bronze = spark.table('googl_stock_data_bronze')
display(apple_table_bronze.head(5))

timestamp,open,high,low,close,volume,symbol,exchange,sector
2025-09-22,254.43,255.78,250.3,252.53,32290538,GOOGL,NASDAQ,COMMUNICATION SERVICES
2025-09-23,253.04,254.36,250.48,251.66,26628016,GOOGL,NASDAQ,COMMUNICATION SERVICES
2025-09-24,251.66,252.3501,246.44,247.14,28201003,GOOGL,NASDAQ,COMMUNICATION SERVICES
2025-09-25,244.4,246.49,240.74,245.79,31020383,GOOGL,NASDAQ,COMMUNICATION SERVICES
2025-09-26,247.065,249.42,245.97,246.54,18503194,GOOGL,NASDAQ,COMMUNICATION SERVICES


In [0]:
# our companies are Apple, Google, Microsoft, and Tesla
time.sleep(30)
update_table('MSFT')
apple_table_bronze = spark.table('msft_stock_data_bronze')
display(apple_table_bronze.head(5))

timestamp,open,high,low,close,volume,symbol,exchange,sector
2025-09-22,515.59,517.74,512.545,514.45,20009314,MSFT,NASDAQ,TECHNOLOGY
2025-09-23,513.8,514.5899,507.31,509.23,19799580,MSFT,NASDAQ,TECHNOLOGY
2025-09-24,510.38,512.48,506.92,510.15,13533711,MSFT,NASDAQ,TECHNOLOGY
2025-09-25,508.3,510.01,505.04,507.03,15786468,MSFT,NASDAQ,TECHNOLOGY
2025-09-26,510.06,513.94,506.62,511.46,16213129,MSFT,NASDAQ,TECHNOLOGY


In [0]:
# our companies are Apple, Google, Microsoft, and Tesla
time.sleep(30)
update_table('TSLA')
apple_table_bronze = spark.table('tsla_stock_data_bronze')
display(apple_table_bronze.head(5))

timestamp,open,high,low,close,volume,symbol,exchange,sector
2025-09-22,431.11,444.98,429.13,434.21,97108777,TSLA,NASDAQ,CONSUMER CYCLICAL
2025-09-23,439.88,440.97,423.72,425.85,83422691,TSLA,NASDAQ,CONSUMER CYCLICAL
2025-09-24,429.83,444.21,429.0301,442.79,93133570,TSLA,NASDAQ,CONSUMER CYCLICAL
2025-09-25,435.24,435.35,419.08,423.39,96746426,TSLA,NASDAQ,CONSUMER CYCLICAL
2025-09-26,428.3,440.47,421.02,440.4,101628160,TSLA,NASDAQ,CONSUMER CYCLICAL
